# E3-10A — Persistence et Ridge : évaluation multihorizons

**Extension indépendante du protocole historique 08A.** Entraînement Ridge sur la partition `train` complète de chaque horizon, sélection de `alpha` uniquement sur `validation`, puis évaluation sur les **13 643 clés de test communes**. Persistence utilise la dernière température observée. Les fichiers historiques ne sont jamais modifiés.

**Important :** l'évaluation sur les clés communes ne signifie pas que les modèles ont été entraînés sur des jeux identiques entre horizons. Le score `1 − RMSE_modèle/RMSE_persistance` est positif si le modèle surpasse la persistance. Les valeurs de R² peuvent être négatives.


In [ ]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path.cwd().resolve()
REP = ROOT/'outputs/E3_aligned_representations'
COMMON = ROOT/'outputs/E3_common_evaluation'
OUT = ROOT/'outputs/E3_10A_persistence_ridge_multihorizon'
OUT.mkdir(parents=True, exist_ok=True)
HORIZONS = [15,30,60,120]
SPLITS = ['train','validation','test']
ALPHAS = [0.01,0.1,1.0,10.0,100.0]  # exact grid from historical 08A
KEYS = ['uuid','timestamp','split']
assert REP.exists() and COMMON.exists(), 'Run E3_07ABCD and E3_08 first.'
common = pd.read_csv(COMMON/'01_common_keys_all_horizons.csv')
common['timestamp'] = pd.to_datetime(common['timestamp'],utc=True)
assert len(common)==91763 and not common.duplicated(KEYS).any()
expected = {'train':64393,'validation':13727,'test':13643}
assert common.groupby('split').size().to_dict()==expected
print('Output directory:',OUT)


In [ ]:
def get_split(h, split):
    folder=REP/f'h{h:03d}'
    meta=pd.read_csv(folder/f'metadata_{split}.csv')
    meta['timestamp']=pd.to_datetime(meta['timestamp'],utc=True)
    meta['target_timestamp']=pd.to_datetime(meta['target_timestamp'],utc=True)
    assert not meta.duplicated(KEYS).any()
    assert meta['split'].eq(split).all()
    assert (meta['target_timestamp']-meta['timestamp']).eq(pd.Timedelta(minutes=h)).all()
    x=np.load(folder/f'X_tabular_{split}_scaled.npy',mmap_mode='r')
    xu=np.load(folder/f'X_tabular_{split}_unscaled.npy',mmap_mode='r')
    y=np.load(folder/f'y_tabular_{split}.npy',mmap_mode='r')
    assert len(meta)==len(x)==len(xu)==len(y) and x.shape[1]==15
    assert np.isfinite(x).all() and np.isfinite(y).all()
    assert np.allclose(meta['target'].to_numpy(),y,rtol=0,atol=1e-5)
    # Layout from E3_07ABCD: 5 history steps, variables [T,RH,CO2], oldest first.
    current_temperature=np.asarray(xu[:,12],dtype=np.float64)
    assert np.allclose(xu.reshape(-1,5,3)[:,-1,0],current_temperature)
    idx=np.load(COMMON/f'indices_h{h:03d}_{split}.npy')
    selected=meta.iloc[idx].reset_index(drop=True)
    wanted=common.loc[common['split'].eq(split),KEYS].copy()
    wanted=wanted.sort_values(KEYS).reset_index(drop=True)
    actual=selected[KEYS].sort_values(KEYS).reset_index(drop=True)
    assert wanted.equals(actual),f'Common key mismatch: {h} {split}'
    return meta,np.asarray(x),np.asarray(y),current_temperature,np.asarray(idx,dtype=np.int64)

def metrics(y,p):
    return dict(n=int(len(y)),MAE=float(mean_absolute_error(y,p)),
                RMSE=float(np.sqrt(mean_squared_error(y,p))),
                R2=float(r2_score(y,p)),Bias=float(np.mean(p-y)))


In [ ]:
all_metrics=[]; all_search=[]; all_predictions=[]; all_devices=[]; configs=[]
for h in HORIZONS:
    print(f'\n--- Horizon {h} minutes ---')
    data={s:get_split(h,s) for s in SPLITS}
    mtr,xtr,ytr,ttr,itr=data['train']
    mva,xva,yva,tva,iva=data['validation']
    mte,xte,yte,tte,ite=data['test']
    # Validation-only hyperparameter selection; no test data used here.
    grid=[]
    for alpha in ALPHAS:
        candidate=Ridge(alpha=alpha)
        candidate.fit(xtr,ytr)
        v=candidate.predict(xva)
        grid.append(dict(horizon_min=h,alpha=alpha,validation_RMSE=metrics(yva,v)['RMSE']))
    all_search.extend(grid)
    best=sorted(grid,key=lambda r:(r['validation_RMSE'],r['alpha']))[0]['alpha']
    start=time.perf_counter()
    model=Ridge(alpha=best).fit(xtr,ytr)
    duration=time.perf_counter()-start
    configs.append(dict(horizon_min=h,alpha=best,train_seconds=duration,
                        train_n=len(ytr),validation_n=len(yva),test_n=len(yte)))
    for split in SPLITS:
        meta,x,y,temp,indices=data[split]
        pred_r=model.predict(x)
        pred_p=temp
        # Train/validation/test: full split metrics (descriptive).
        for name,pred in [('Persistence',pred_p),('Ridge',pred_r)]:
            all_metrics.append(dict(horizon_min=h,split=split,scope='full_horizon_split',
                                    model=name,alpha=(best if name=='Ridge' else np.nan),
                                    **metrics(y,pred)))
        # Common-key metrics: primary paired comparison (test).
        yc=y[indices]
        pc=pred_p[indices]
        rc=pred_r[indices]
        p_rmse=metrics(yc,pc)['RMSE']
        for name,pred in [('Persistence',pc),('Ridge',rc)]:
            stat=metrics(yc,pred)
            all_metrics.append(dict(horizon_min=h,split=split,scope='common_keys',
                                    model=name,alpha=(best if name=='Ridge' else np.nan),
                                    skill_RMSE=(1-stat['RMSE']/p_rmse if p_rmse else np.nan),**stat))
            if split=='test':
                frame=meta.iloc[indices][['uuid','timestamp','target_timestamp','split']].copy()
                frame['horizon_min']=h
                frame['model']=name
                frame['y_true']=yc
                frame['y_pred']=pred
                frame['error']=pred-yc
                all_predictions.append(frame)
                for uuid,g in frame.groupby('uuid'):
                    st=metrics(g['y_true'],g['y_pred'])
                    all_devices.append(dict(horizon_min=h,model=name,uuid=uuid,**st))
    print('alpha:',best,'common TEST persistence RMSE:',round(metrics(yte[ite],tte[ite])['RMSE'],6),
          'ridge RMSE:',round(metrics(yte[ite],model.predict(xte)[ite])['RMSE'],6))


In [ ]:
metrics_df=pd.DataFrame(all_metrics)
search_df=pd.DataFrame(all_search)
pred_df=pd.concat(all_predictions,ignore_index=True)
per_device_df=pd.DataFrame(all_devices)
config_df=pd.DataFrame(configs)
metrics_df.to_csv(OUT/'01_metrics_all_splits.csv',index=False)
search_df.to_csv(OUT/'02_ridge_validation_search.csv',index=False)
pred_df.to_csv(OUT/'03_common_test_predictions.csv',index=False)
per_device_df.to_csv(OUT/'04_common_test_per_device.csv',index=False)
config_df.to_csv(OUT/'05_selected_alpha_and_training.csv',index=False)
primary=metrics_df.query("split=='test' and scope=='common_keys'").copy()
primary=primary[['horizon_min','model','n','RMSE','MAE','R2','Bias','skill_RMSE','alpha']]
primary.to_csv(OUT/'00_PRIMARY_common_TEST_results.csv',index=False)
assert primary.groupby('horizon_min')['n'].apply(lambda z:z.eq(13643).all()).all()
assert not pred_df.duplicated(['uuid','timestamp','horizon_min','model']).any()
assert pred_df.groupby(['horizon_min','model']).size().eq(13643).all()
print(primary.to_string(index=False))
print('Saved outputs to:',OUT)


## Interprétation et limites

- Un `skill_RMSE` positif signifie une amélioration par rapport à la persistance, calculée sur les **mêmes clés** pour chaque horizon.
- La comparaison de RMSE entre horizons reste descriptive : les cibles sont différentes, même si les origines temporelles sont identiques.
- Les modèles Ridge sont ajustés sur les **ensembles complets spécifiques à chaque horizon** et sélectionnés uniquement sur validation, comme dans le protocole historique 08A. Les scores historiques à 15 minutes ne sont **pas remplacés**.
- Ce notebook ne fournit pas encore d'intervalles de confiance appariés, de rolling-origin evaluation ni de comparaisons entre architectures profondes.
- **À transmettre :** `00_PRIMARY_common_TEST_results.csv`, `02_ridge_validation_search.csv` et, si possible, `04_common_test_per_device.csv`.
